# Computational Theory: SHA-256

In [18]:
import numpy as np
import hashlib
import sys

## Problem 1: Representing SHA-256 Data

### 32-bit word

SHA-256 works on 32-bit words (FIPS 180-4 Section 1 Figure 1), with addition modulo 2³² (Section 3.2). A word here is a `np.uint32` and because it has a fixed width, addition wraps on its own without a mask. `np.errstate(over="ignore")` turns off the overflow warning that NumPy gives when a scalar wraps because the wrap is required by Section 3.2, not a bug being hidden. `~` also flips exactly 32 bits, which Ch (Section 4.1.2) needs. On a plain Python int, `~0` gives `-1` instead. Plain ints that are too large for 32 bits raise an `OverflowError`, which is why they are created with `np.uint32()`.

#### References
[NIST, *FIPS 180-4: Secure Hash Standard*](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

[NumPy, Data types: Overflow errors](https://numpy.org/doc/stable/user/basics.types.html#overflow-errors)

In [19]:
# H0, the first word of the initial hash value
w = np.uint32(0x6A09E667)
# Prints the word as 8 hex digits/32 bits
print(f"{w:08x}")

# Shows that addition wraps around at 2^32
with np.errstate(over="ignore"):
    wrapped = np.uint32(0xFFFFFFFF) + np.uint32(1)
print("After wrap:", hex(wrapped))
assert wrapped == 0

# Ch (FIPS 180-4 Section 4.1.2) uses NOT; ~ on uint32 flips all 32 bits, 
# but on a plain int it goes negative
print("~ uint32:", hex(~np.uint32(0)))
print("~ int   :", ~0)

6a09e667
After wrap: 0x0
~ uint32: 0xffffffff
~ int   : -1


### Sequences of 32-bit words
SHA-256 handles words in fixed-length groups. The initial hash value H⁽⁰⁾ is 8 words (FIPS 180-4 Section 5.3.3) and the round constants K are 64 words (Section 4.2.2). The message schedule W is 64 words (Section 6.2.2), and a 512-bit message block is 16 words (Section 5.2.1). A sequence is a NumPy array with `dtype=np.uint32`, so each element is a 32-bit word.

An array suits this better than a Python list for two reasons. First, `+` means something different. On lists, `[1, 2] + [3, 4]` joins them end to end and gives `[1, 2, 3, 4]`. On arrays, `+` adds element by element, which is what the standard means when it adds two sequences of words, for example when it updates the hash value (Section 6.2.2). Array addition also wraps each word modulo 2³² without a warning, so `np.errstate` isn't needed. Second, an array's size is set when it is created. That matches the fixed sizes above, and a sequence can't grow or shrink by accident.

#### References
[NIST, *FIPS 180-4: Secure Hash Standard*](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

[NumPy, *the absolute basics for beginners*](https://numpy.org/doc/stable/user/absolute_beginners.html)

In [20]:
# H(0), the initial hash value (Section 5.3.3), is 8 words
H = np.array([0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
              0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19],
             dtype=np.uint32)
# Format each word as 8 hex digits (32 bits), separated by spaces
print(" ".join(f"{h:08x}" for h in H))
# 8 words x 4 bytes each = 32 bytes = 256 bits, the size of a SHA-256 hash
assert H.nbytes * 8 == 256

# + joins lists but adds arrays element by element
print("list  +:", [1, 2] + [3, 4])
a = np.array([1, 2], dtype=np.uint32)
b = np.array([3, 4], dtype=np.uint32)
print("array +:", a + b)

# Array addition wraps each word at 2^32
a = np.array([0xFFFFFFFF, 1], dtype=np.uint32)
b = np.array([1, 1], dtype=np.uint32)
# 0xFFFFFFFF + 1 = 2^32, which doesn't fit in 32 bits, so it wraps to 0
# 1 + 1 = 2 is unaffected
print("a + b:", [hex(x) for x in a + b])
assert list(a + b) == [0, 2]

6a09e667 bb67ae85 3c6ef372 a54ff53a 510e527f 9b05688c 1f83d9ab 5be0cd19
list  +: [1, 2, 3, 4]
array +: [4 6]
a + b: ['0x0', '0x2']


### Input messages

FIPS 180-4 describes a message as a string of bits of any length. For SHA-256 it must be shorter than 2⁶⁴ bits (Section 1). Python stores data in whole bytes, so a message is a `bytes` object: a sequence of integers from 0 to 255. Text isn't bytes, so it has to be encoded first with `.encode("utf-8")`. Under UTF-8, one character can become more than one byte. For example, `"€"` becomes 3 bytes. A `bytes` object is immutable, meaning it can't be changed after it is made, so the message stays exactly as it was given.

#### References
[NIST, *FIPS 180-4: Secure Hash Standard*](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

[Python, *Binary Sequence Types*](https://docs.python.org/3/library/stdtypes.html#binary-sequence-types-bytes-bytearray-memoryview)

In [21]:
# "abc" encoded to bytes, the standard SHA-256 test message
msg = "abc".encode("utf-8")
print("abc:", msg.hex(" "), "|", len(msg) * 8, "bits")
assert len(msg) * 8 == 24

# One character can take more than one byte
euro = "€".encode("utf-8")
print("€  :", euro.hex(" "), "|", len(euro), "bytes")
assert len(euro) == 3

abc: 61 62 63 | 24 bits
€  : e2 82 ac | 3 bytes


### 512-bit message blocks

SHA-256 processes a padded message in 512-bit blocks. Each block is 64 bytes, split into sixteen 32-bit words (Section 5.2.1), so a block is a NumPy array of 16 `np.uint32` values. `np.frombuffer` with `dtype=">u4"` reads each group of 4 bytes as a big-endian word, with the first byte as the most significant, as the standard requires (Section 3.1). Without this, a little-endian machine would read `61 62 63 64` as `0x64636261` instead of `0x61626364`, and every hash would be wrong. `.astype(np.uint32)` keeps the same values but turns the read-only `>u4` array from `frombuffer` into a writable `np.uint32` copy, so blocks match every other word array in the notebook and can be used to build the message schedule W.

#### References
[NIST, *FIPS 180-4: Secure Hash Standard*](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

[NumPy, *numpy.frombuffer*](https://numpy.org/doc/stable/reference/generated/numpy.frombuffer.html)

In [22]:
# One 64-byte block read as 16 big-endian 32-bit words
block = np.frombuffer(bytes(range(64)), dtype=">u4").astype(np.uint32)
print(len(block), "words:", " ".join(f"{w:08x}" for w in block[:4]), "...")
assert len(block) == 16 and block.nbytes * 8 == 512

# Byte order matters: the same 4 bytes read both ways
print(f"big-endian   : {np.frombuffer(b'abcd', dtype='>u4')[0]:08x}")
print(f"little-endian: {np.frombuffer(b'abcd', dtype='<u4')[0]:08x}")

16 words: 00010203 04050607 08090a0b 0c0d0e0f ...
big-endian   : 61626364
little-endian: 64636261


### The final 256-bit hash value

While computing, the hash is the same 8-word `np.uint32` array as H⁽⁰⁾ in the sequences section. After the last block, the 8 words are joined end to end in big-endian order into 256 bits, or 32 bytes (Section 6.2.2). Convert to `>u4` before calling `.tobytes()`, because plain `np.uint32` on a little-endian machine writes each word's bytes in reverse. The result is usually shown as 64 hex characters, such as `ba7816bf8f01cfea…` for "abc".

#### References
[NIST, *FIPS 180-4: Secure Hash Standard*](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

[Python, *hashlib — Secure hashes and message digests*](https://docs.python.org/3/library/hashlib.html)

In [23]:
# Known answer for "abc", read as 8 big-endian 32-bit words
abc_expected = hashlib.sha256(b"abc").digest()
H_abc = np.frombuffer(abc_expected, dtype=">u4").astype(np.uint32)
print("8 words:", " ".join(f"{w:08x}" for w in H_abc))

# Convert back to 32 bytes with >u4 and compare with hashlib
abc_digest = H_abc.astype(">u4").tobytes()
print("hex    :", abc_digest.hex())
assert abc_digest.hex() == hashlib.sha256(b"abc").hexdigest()

8 words: ba7816bf 8f01cfea 414140de 5dae2223 b00361a3 96177a9c b410ff61 f20015ad
hex    : ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


### Big-endian integers

Big-endian means the most significant byte comes first (Section 3.1). SHA-256 uses it everywhere a number is turned into bytes or bytes into a number. It appears in SHA-256 when reading block words and writing the digest, as shown in the 512-bit message blocks and final hash value sections, and in the 64-bit message length added during padding (Section 5.1.1). Python handles it using NumPy's `>u4` for an array of words and `int.from_bytes(..., "big")` / `n.to_bytes(8, "big")` for a single larger number like 64-bit length. >u4 only covers 32-bit words, and the message length is 64 bits, so it needs to_bytes(8, "big") instead.

#### References
[NIST, *FIPS 180-4: Secure Hash Standard*](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

[Python, *int.to_bytes*](https://docs.python.org/3/library/stdtypes.html#int.to_bytes)

[Python, *int.from_bytes*](https://docs.python.org/3/library/stdtypes.html#int.from_bytes)

In [24]:
# Show computer is little endian, so byte order has to be stated explicitly
print("byte order:", sys.byteorder)

# Shows the same 4 bytes matching the >u4 result in the blocks cell
print(hex(int.from_bytes(b"abcd", "big")))
assert int.from_bytes(b"abcd", "big") == 0x61626364

# Show the 64-bit length of "abc" in bytes, as required by SHA-256 padding
abc_length = (len(msg) * 8).to_bytes(8, "big")
print("64-bit length of abc:", abc_length.hex())
assert abc_length.hex() == "0000000000000018"


byte order: little
0x61626364
64-bit length of abc: 0000000000000018


## Problem 2: SHA-256 Bitwise Operations

SHA-256 uses six functions (Section 4.1.2) that work on 32-bit `np.uint32` words, as set up in Problem 1. Since four of the six functions are built from rotate right (ROTR) and shift right (SHR), defined in Section 3.2, these are written once below as helpers. The six functions mix the bits of each word, so that changing one input bit changes many output bits. σ₀ and σ₁ build the message schedule, while Σ₀, Σ₁, Ch and Maj are used in each of the 64 compression rounds (Section 6.2.2).

#### References 

[NIST, *FIPS 180-4: Secure Hash Standard*](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)

In [25]:
def SHR(x, n):
    """Shift a 32-bit word right by n positions.

    FIPS 180-4 Section 3.2.
    x: the 32-bit word to shift, as np.uint32.
    n: the number of positions to shift right, an integer with 0 <= n < 32.
    Returns: x >> n as np.uint32; the n low bits are dropped and zeros fill
    in from the top.
    """
    return x >> n


def ROTR(x, n):
    """Rotate a 32-bit word right by n positions.

    FIPS 180-4 Section 3.2.
    x: the 32-bit word to rotate, as np.uint32.
    n: the number of positions to rotate right, an integer with 0 <= n <= 32.
    Returns: (x >> n) | (x << (32 - n)) as np.uint32; the n low bits wrap round
    to the top.
    """
    return (x >> n) | (x << (32 - n))


assert ROTR(np.uint32(1), 1) == 0x80000000
assert SHR(np.uint32(1), 1) == 0 

test_words = [np.uint32(0x00000000), np.uint32(0xFFFFFFFF),
              np.uint32(0x12345678), np.uint32(0x87654321)]
for x in test_words:
    assert ROTR(x, 0) == x
    assert ROTR(x, 32) == x

## Problem 3: Generating the SHA-256 Constants

## Problem 4: Padding and Parsing Messages

## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256